# BirdCLEF: offline CPU inference

Attach the **BirdCLEF+ 2026 competition**, the [**birdclef-2026-wheelhouse Dataset**](https://www.kaggle.com/datasets/denysmykhailov/birdclef-2026-wheelhouse),
and either the [**baseline models Dataset**](https://www.kaggle.com/datasets/denysmykhailov/birdclef-2026-models), another **Dataset with model runs**, or the **saved Output of a training notebook**.
Select **Accelerator: None** and **Internet: OFF**. Run All, then Save Version / Save & Run All.

A run folder contains `model-config.json`, `classes.json`, `fit.json` and the best
checkpoint referenced in `fit.json`. Training notebook outputs already have this
layout. For local models, upload `build/birdclef-models.zip` as a Dataset.

Select one run below for a baseline. Select several to average their probabilities;
`WEIGHTS=None` means equal weights. No W&B, downloads or training during inference.

In [ ]:
from pathlib import Path

wheel, = [
    path for path in Path("/kaggle/input").rglob("birdclef*.whl")
    if (path.parent / "requirements.txt").is_file()
]
bundle = wheel.parent
%pip install --no-index --find-links "{bundle}" -r "{bundle}/requirements.txt" -q
%pip install --no-index --force-reinstall --no-deps "{wheel}" -q

In [ ]:
import os

CPU_THREADS = 4
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = str(CPU_THREADS)

available_runs = sorted(path.parent for path in Path("/kaggle/input").rglob("fit.json"))
for run in available_runs:
    print(run)
assert available_runs, "Attach a model Dataset or the saved Output of a training notebook."

## Choose models

The default selects the first printed run. Replace the list with explicit paths
printed above to use another run or several runs. Both Dataset and notebook Output
paths work. For example, `RUN_DIRS = [available_runs[1]]` selects the second model,
and `RUN_DIRS = available_runs[:2]` averages the first two models.
Optional `WEIGHTS = [0.7, 0.3]` must match their order.

Keep smoke runs out of the selected list. For separate baseline submissions, save
one notebook version per model.

In [ ]:
RUN_DIRS = [available_runs[0]]
WEIGHTS = None
BATCH_SIZE = 32
NUM_WORKERS = 0

COMPETITION = next(
    path.parent for path in Path("/kaggle/input").rglob("sample_submission.csv")
    if path.parent.name == "birdclef-2026"
)
OUTPUT = Path("/kaggle/working/submission.csv")

In [ ]:
import torch
from birdclef.submission import run_inference

torch.set_num_threads(CPU_THREADS)
run_inference(
    RUN_DIRS, COMPETITION, OUTPUT,
    weights=WEIGHTS, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)

## Output

During an ordinary Run All, hidden test audio is absent. The code checks all selected
models on one real training minute and saves `smoke-predictions.csv`; `submission.csv`
then contains the sample rows so the saved notebook has an output to submit.

During scoring, it predicts **all actual test soundscapes**, builds IDs from their
filenames and five-second end times, and writes the real `submission.csv`.
The public smoke run checks execution and formatting, not leaderboard quality or
full hidden-test runtime.

In [ ]:
import pandas as pd

submission = pd.read_csv(OUTPUT)
print(f"{OUTPUT}: {submission.shape[0]} rows, {submission.shape[1] - 1} classes")
submission.head()